In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# بررسی بیدار بودن گرافیک
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# خواندن فایل
df = pd.read_csv('digikala-comments.csv')

# ترکیب عنوان و متن
df['full_text'] = df['title'].fillna('') + " " + df['body'].fillna('')
df = df.dropna(subset=['recommendation_status'])

# استخراج هر ۳ کلاس
df = df[df['recommendation_status'].isin(['recommended', 'not_recommended', 'no_idea'])]

# اختصاص اعداد: 0=منفی ، 1=خنثی ، 2=مثبت
def get_label(status):
    if status == 'recommended': return 2
    elif status == 'no_idea': return 1
    else: return 0
    
df['sentiment'] = df['recommendation_status'].apply(get_label)

# متوازن‌سازی (استفاده از replace=True برای جلوگیری از خطا در صورت کم بودن داده‌های خنثی)
df_pos = df[df['sentiment'] == 2].sample(n=3500, random_state=42, replace=True)
df_neu = df[df['sentiment'] == 1].sample(n=3500, random_state=42, replace=True)
df_neg = df[df['sentiment'] == 0].sample(n=3500, random_state=42, replace=True)
df_balanced = pd.concat([df_pos, df_neu, df_neg])

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced['full_text'].tolist(), 
    df_balanced['sentiment'].tolist(), 
    test_size=0.2, 
    random_state=42
)
print(f"Total 3-Class balanced dataset: {len(df_balanced)} records")

In [ ]:
# 🌐 دریافت مستقیم توکنایزر از سرورهای هاگینگ‌فیس (Hugging Face)
model_name = "HooshvareLab/bert-fa-base-uncased"

print("Downloading Tokenizer from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

class DigikalaDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DigikalaDataset(train_encodings, train_labels)
test_dataset = DigikalaDataset(test_encodings, test_labels)
print("✅ Tokenization Complete!")

In [ ]:
print("Downloading Model Weights from Hugging Face...")

# عدد num_labels روی 3 تنظیم شده و مدل مستقیماً از اینترنت دریافت می‌شود
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

print(f"✅ Model (3-Classes) successfully downloaded and loaded into {device}!")

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",  
    save_strategy="epoch",
    fp16=True, 
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("🚀 Starting Offline Fine-Tuning...")
trainer.train()

In [ ]:
# مدلی که الان آموزش دید را در یک پوشه جدید ذخیره می‌کنیم
final_model_path = "./dl_sentiment_model"

model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"🎯 Training Complete! Fine-tuned model saved to {final_model_path}")

In [ ]:
import json

# ذخیره اطلاعات واقعی ارزیابی اپوک دوم (بهترین حالت)
model_metrics = {
    "accuracy": 74.1905,
    "best_epoch": 2,
    "architecture": "ParsBERT v2 (3-Class)"
}

# نوشتن فایل جی‌سان در پوشه مدل
with open("./dl_sentiment_model/metrics.json", "w", encoding="utf-8") as f:
    json.dump(model_metrics, f, ensure_ascii=False, indent=4)

print("✅ فایل شناسنامه پویا با موفقیت ساخته شد.")